<a href="https://colab.research.google.com/github/Seifeddin84/SISCOIN/blob/main/New_PINN_15102025_analyzer%20and%20predictor%20PINNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 78.2 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
# -*- coding: utf-8 -*-
"""
PINN_Specialist_Batch_Analysis_v4.py

This version is optimized for LARGE datasets (~2000 files).
1. Corrects the analysis logic to process ALL files independently (no train/test split).
2. Implements a high-performance PyTorch Dataset and DataLoader with parallel workers
   to eliminate the CPU/IO bottleneck and maximize GPU utilization.
"""
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import os
import re
from sklearn.metrics import mean_squared_error, r2_score
# NEW: Import Dataset and DataLoader
from torch.utils.data import Dataset, DataLoader

# ============================================================================
# STEP 1: DATA LOADER (UNCHANGED, still provides data in SI units)
# ============================================================================
def load_balance_data(file_path):
    record = wfdb.rdrecord(file_path)
    data = pd.DataFrame(record.p_signal, columns=record.sig_name)
    sample_rate = record.fs
    subject_info = {'weight': 75.0, 'height': 1.70, 'record_name': os.path.basename(file_path)}

    with open(file_path + '.hea', 'r') as f:
        content = f.read()
        height_match = re.search(r'Height:\s*(\d+\.?\d*)', content)
        weight_match = re.search(r'Weight:\s*(\d+\.?\d*)', content)

        if height_match:
            subject_info['height'] = float(height_match.group(1)) * 0.55 / 100
        if weight_match:
            subject_info['weight'] = float(weight_match.group(1))

    time = np.linspace(0, len(data) / sample_rate, len(data))
    cop_data = data[['COPx', 'COPy']].values / 100.0 # Convert cm to METERS
    return time, cop_data, subject_info

# ============================================================================
# NEW: PYTORCH DATASET FOR EFFICIENT LOADING
# ============================================================================
class BalanceDataset(Dataset):
    """A custom PyTorch Dataset to handle loading balance files."""
    def __init__(self, folder_path, limit_files=None):
        self.file_paths = [os.path.join(folder_path, f[:-4]) for f in os.listdir(folder_path) if f.endswith('.dat')]
        if limit_files:
            self.file_paths = self.file_paths[:limit_files]
        print(f"Dataset initialized with {len(self.file_paths)} files.")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        try:
            time, cop_data, subject_info = load_balance_data(file_path)
            # We need to return tensors, so we convert them here
            return {
                'time': torch.tensor(time, dtype=torch.float32),
                'cop_data': torch.tensor(cop_data, dtype=torch.float32),
                'weight': torch.tensor(subject_info['weight'], dtype=torch.float32),
                'height': torch.tensor(subject_info['height'], dtype=torch.float32),
                'record_name': subject_info['record_name']
            }
        except Exception as e:
            print(f"Warning: Could not load {file_path}. Error: {e}")
            return None

# ============================================================================
# PINN MODEL AND PHYSICS (UNCHANGED from v3)
# ============================================================================
class SimpleBalancePINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, 100), nn.Tanh(),
            nn.Linear(100, 100), nn.Tanh(),
            nn.Linear(100, 2)
        )
        self.layers.apply(self._init_weights)
        self.log_damping = nn.Parameter(torch.tensor(np.log(20.0), dtype=torch.float32))
        self.log_stiffness = nn.Parameter(torch.tensor(np.log(500.0), dtype=torch.float32))
    def _init_weights(self, module):
        if isinstance(module, nn.Linear): nn.init.xavier_normal_(module.weight)
    @property
    def damping(self): return torch.exp(self.log_damping)
    @property
    def stiffness(self): return torch.exp(self.log_stiffness)
    def forward(self, t): return self.layers(t)

def physics_loss(model, time_points, mass, height):
    time_points.requires_grad_(True)
    position = model(time_points)
    velocity = torch.autograd.grad(position.sum(), time_points, create_graph=True)[0]
    acceleration = torch.autograd.grad(velocity.sum(), time_points, create_graph=True)[0]
    gravity = 9.81
    term1 = mass * height * acceleration
    term2 = (model.damping / height) * velocity
    term3 = (model.stiffness / height - mass * gravity) * position
    physics_residual = term1 + term2 + term3
    return torch.mean(physics_residual**2)

def parameter_regularization_loss(model):
    damping_penalty = torch.relu(model.damping - 50.0)**2 + torch.relu(1.0 - model.damping)**2
    stiffness_penalty = torch.relu(model.stiffness - 1000.0)**2 + torch.relu(100.0 - model.stiffness)**2
    return damping_penalty + stiffness_penalty

# ============================================================================
# TRAINING LOGIC (UNCHANGED from v3)
# ============================================================================
def train_simple_pinn(t_tensor, cop_tensor, weight, height, epochs=8000): # MODIFIED: epochs=8000
    model = SimpleBalancePINN()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    damping_history, stiffness_history = [], []

    print("Starting training...")
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(t_tensor)
        data_loss = nn.MSELoss()(predictions, cop_tensor)
        phys_loss = physics_loss(model, t_tensor, weight, height)
        reg_loss = parameter_regularization_loss(model)
        lambda_physics, lambda_reg = 0.01, 0.1
        total_loss = data_loss + lambda_physics * phys_loss + lambda_reg * reg_loss
        total_loss.backward()
        optimizer.step()
        damping_history.append(model.damping.item())
        stiffness_history.append(model.stiffness.item())
        if epoch % 2000 == 0:
            print(f"  Epoch {epoch}/{epochs}: Loss={total_loss:.6f}, K={model.stiffness.item():.2f}, C={model.damping.item():.2f}")
    print("Training completed!")
    return model, damping_history, stiffness_history

# ============================================================================
# PLOTTING FUNCTIONS (UNCHANGED from v3)
# ============================================================================
# analyze_and_plot_single_subject, plot_parameter_evolution, and
# plot_population_summary functions remain the same as the previous version.
# (Code omitted for brevity, but they should be included in your final script)
def analyze_and_plot_single_subject(model, time, cop_data, subject_info, record_name):
    # ... (same plotting code as before)
    pass
def plot_parameter_evolution(damping_history, stiffness_history, record_name):
    # ... (same plotting code as before)
    pass
def plot_population_summary(all_mass, all_height, all_damping, all_stiffness):
    # ... (same plotting code as before)
    pass

# ============================================================================
# MAIN BATCH-PROCESSING LOOP (MODIFIED FOR DATALOADER)
# ============================================================================
def analyze_multiple_files(folder_path, limit_files=None, num_workers=4):
    """
    MODIFIED: This function now uses the efficient DataLoader to process all files.
    """
    dataset = BalanceDataset(folder_path, limit_files=limit_files)
    # Use batch_size=1 because each subject is a separate experiment.
    # num_workers > 0 tells the DataLoader to use background processes for loading.
    data_loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=num_workers)

    all_mass, all_height, all_damping, all_stiffness = [], [], [], []

    for i, batch in enumerate(data_loader):
        if batch is None: continue # Skip if a file failed to load

        # Unpack the batch (batch_size is 1, so we squeeze the dimensions)
        t_tensor = batch['time'].squeeze(0).view(-1, 1)
        cop_tensor = batch['cop_data'].squeeze(0)
        weight = batch['weight'].item()
        height = batch['height'].item()
        record_name = batch['record_name'][0]

        print(f"\n{'='*60}\nProcessing file {i+1}/{len(dataset)}: {record_name}\n{'='*60}")
        print(f"  Subject Info: {weight:.1f} kg, {height:.2f} m COM height")

        try:
            model, damp_hist, stiff_hist = train_simple_pinn(t_tensor, cop_tensor, weight, height, epochs=8000)

            # Store results
            all_mass.append(weight)
            all_height.append(height)
            all_damping.append(model.damping.item())
            all_stiffness.append(model.stiffness.item())

            # Generate plots (ensure you have the plotting functions from the previous script)
            # analyze_and_plot_single_subject(model, t_tensor.numpy(), cop_tensor.numpy(), {}, record_name)
            # plot_parameter_evolution(damp_hist, stiff_hist, record_name)

            print(f"✓ Successfully processed {record_name}")

        except Exception as e:
            print(f"✗ Error processing {record_name}: {e}")
            import traceback
            traceback.print_exc()


    return all_mass, all_height, all_damping, all_stiffness

if __name__ == "__main__":
    data_folder = "/content/drive/MyDrive/human-balance-evaluation-database-1.0.0"

    if os.path.exists(data_folder):
        # We analyze ALL files. Set limit_files=None to run on the full 1938 files.
        # num_workers should be > 0. A good starting point is half your CPU cores.
        mass, height, damping, stiffness = analyze_multiple_files(
            data_folder,
            limit_files=5, # Set to None to run on all 1938 files
            num_workers=4     # Adjust based on your CPU cores
        )

        if mass:
            # plot_population_summary(mass, height, damping, stiffness)
            print("\nAnalysis of all files complete. Summary data collected.")
        else:
            print("No data was successfully processed.")
    else:
        print(f"ERROR: Data folder not found at '{data_folder}'")

Dataset initialized with 5 files.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



Processing file 1/5: BDS01002
  Subject Info: 67.9 kg, 0.85 m COM height
Starting training...
  Epoch 0/8000: Loss=11.011180, K=500.50, C=19.98
  Epoch 2000/8000: Loss=0.000921, K=516.55, C=20.07
  Epoch 4000/8000: Loss=0.000891, K=527.51, C=21.17
  Epoch 6000/8000: Loss=0.000776, K=545.75, C=19.76
Training completed!
✓ Successfully processed BDS01002

Processing file 2/5: BDS01047
  Subject Info: 54.5 kg, 0.77 m COM height
Starting training...
  Epoch 0/8000: Loss=390.885529, K=499.50, C=19.98
